# Capítulo 11 · Proyecto integrador: un generador de relatos

Datos creados con una gramática, un GPT de caracteres, entrenamiento con método, generación con temperatura y top-p, evaluación con criterios explícitos y fichas de datos y del modelo. PyTorch en CPU; no necesita descargar nada. El experimento completo tarda unos dieciséis minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Constantes y datos

In [1]:
"""Capítulo 11: proyecto integrador, un generador de relatos. CPU."""
import math
import random
import re
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

SEMILLA = 1337
RELATOS = 8000
CONFIG = dict(contexto=256, dimension=64, cabezas=4, capas=4, lote=16)
PASOS, TASA = 2500, 3e-3
CARPETA = Path(".")
SALIDAS = CARPETA / "salidas" / f"contexto_{CONFIG['contexto']}"

# --- 1. Los datos: una gramática de relatos ----------------------------------

ANIMALES = [("una", "tortuga"), ("un", "ratón"), ("una", "ardilla"),
            ("un", "búho"), ("una", "liebre"), ("un", "erizo"),
            ("una", "rana"), ("un", "zorro"), ("una", "cigüeña"),
            ("un", "topo")]
NOMBRES = {"una": ["Lola", "Nina", "Clara", "Luna", "Marta", "Rosa", "Elena",
                   "Sara"],
           "un": ["Pablo", "Tomás", "Hugo", "Lucas", "Mateo", "Bruno", "Dani",
                  "Leo"]}
ADJETIVOS = {"una": ["pequeña", "curiosa", "valiente", "tranquila",
                     "despistada"],
             "un": ["pequeño", "curioso", "valiente", "tranquilo",
                    "despistado"]}
LUGARES = ["el bosque", "la pradera", "la orilla del río", "un viejo molino",
           "la montaña", "un jardín"]
OBJETOS = {"sombrero": "lo", "bufanda": "la", "farol": "lo", "cesta": "la",
           "libro": "lo", "flauta": "la", "llave": "la", "mapa": "lo"}
SITIOS = ["junto al pozo", "bajo un puente", "en una cueva", "entre las rocas",
          "detrás del granero", "en lo alto de un árbol",
          "dentro de una bota vieja"]
AYUDANTES = ["una lechuza sabia", "un caracol paciente",
             "una abeja trabajadora", "un viejo castor", "una golondrina",
             "un perro amable"]
MORALEJAS = ["pedir ayuda no es de débiles", "con paciencia todo se encuentra",
             "hay que cuidar lo que uno quiere",
             "los amigos aparecen cuando más se necesitan",
             "buscar con calma da mejores frutos"]


def relato(azar):
    """Un relato elegido al azar según la gramática."""
    articulo, animal = azar.choice(ANIMALES)
    nombre = azar.choice(NOMBRES[articulo])
    adjetivo = azar.choice(ADJETIVOS[articulo])
    llamado = "llamada" if articulo == "una" else "llamado"
    lugar = azar.choice(LUGARES)
    objeto = azar.choice(list(OBJETOS))
    sitio1, sitio2 = azar.sample(SITIOS, 2)
    inicio = azar.choice([
        f"Había una vez {articulo} {animal} {adjetivo} {llamado} {nombre} "
        f"que vivía en {lugar}.",
        f"En {lugar} vivía {articulo} {animal} {adjetivo} que se llamaba "
        f"{nombre}."])
    problema = azar.choice([
        f"Un día, {nombre} perdió su {objeto}.",
        f"Una mañana, {nombre} no encontraba su {objeto}."])
    return " ".join([
        inicio, problema,
        f"{nombre} {OBJETOS[objeto]} buscó {sitio1}, pero no estaba allí.",
        f"Entonces {azar.choice(AYUDANTES)} le dijo que mirara {sitio2}.",
        f"Allí estaba su {objeto}, y {nombre} aprendió que "
        f"{azar.choice(MORALEJAS)}."])


def crear_datos(n=RELATOS, semilla=SEMILLA):
    """Relatos distintos, repartidos 80/10/10 sin repetir ninguno."""
    azar = random.Random(semilla)
    relatos = []
    vistos = set()
    while len(relatos) < n:
        texto = relato(azar)
        if texto not in vistos:
            vistos.add(texto)
            relatos.append(texto)
    a, b = int(0.8 * n), int(0.9 * n)
    return relatos[:a], relatos[a:b], relatos[b:]

In [2]:
import random
azar = random.Random(0)
for _ in range(2):
    print(relato(azar), "\n")

En la orilla del río vivía una rana pequeña que se llamaba Elena. Una mañana, Elena no encontraba su mapa. Elena lo buscó entre las rocas, pero no estaba allí. Entonces una golondrina le dijo que mirara en una cueva. Allí estaba su mapa, y Elena aprendió que con paciencia todo se encuentra. 

Había una vez una cigüeña valiente llamada Clara que vivía en la pradera. Una mañana, Clara no encontraba su bufanda. Clara la buscó detrás del granero, pero no estaba allí. Entonces una lechuza sabia le dijo que mirara en una cueva. Allí estaba su bufanda, y Clara aprendió que pedir ayuda no es de débiles. 



## 2. El modelo

In [3]:
# --- 2. El modelo de los capítulos 6 a 10 ------------------------------------

class Cabeza(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["contexto"]
        tamano = C // config["cabezas"]
        self.consulta = nn.Linear(C, tamano, bias=False)
        self.clave = nn.Linear(C, tamano, bias=False)
        self.valor = nn.Linear(C, tamano, bias=False)
        self.register_buffer("mascara", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.consulta(x), self.clave(x), self.valor(x)
        puntuaciones = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        futuro = self.mascara[:T, :T] == 0
        puntuaciones = puntuaciones.masked_fill(futuro, float("-inf"))
        return F.softmax(puntuaciones, dim=-1) @ v


class Bloque(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.cabezas = nn.ModuleList(
            [Cabeza(config) for _ in range(config["cabezas"])])
        self.proyeccion = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norma1, self.norma2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norma1(x)
        atencion = torch.cat([cabeza(h) for cabeza in self.cabezas], dim=-1)
        x = x + self.proyeccion(atencion)
        return x + self.mlp(self.norma2(x))


class GPT(nn.Module):
    def __init__(self, vocabulario, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.embedding_simbolos = nn.Embedding(vocabulario, C)
        self.embedding_posiciones = nn.Embedding(config["contexto"], C)
        self.bloques = nn.Sequential(
            *[Bloque(config) for _ in range(config["capas"])])
        self.norma = nn.LayerNorm(C)
        self.salida = nn.Linear(C, vocabulario)

    def forward(self, ids):
        posiciones = torch.arange(ids.shape[1])
        x = self.embedding_simbolos(ids) + self.embedding_posiciones(posiciones)
        return self.salida(self.norma(self.bloques(x)))

## 3. Entrenar

In [4]:
# --- 3. Entrenar (capítulo 7) ------------------------------------------------

def lote(datos, generador):
    T, B = CONFIG["contexto"], CONFIG["lote"]
    inicios = torch.randint(len(datos) - T - 1, (B,), generator=generador)
    x = torch.stack([datos[i:i + T] for i in inicios])
    y = torch.stack([datos[i + 1:i + T + 1] for i in inicios])
    return x, y


def perdida_media(modelo, x, y):
    logits = modelo(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def perdida_texto(modelo, datos, grupo=128):
    """Pérdida media en todo un bloque, en trozos del tamaño del contexto."""
    T = modelo.config["contexto"]
    n = (len(datos) - 1) // T
    x, y = datos[:n * T].view(n, T), datos[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, grupo):
        total += perdida_media(modelo, x[i:i + grupo],
                               y[i:i + grupo]).item() * len(x[i:i + grupo])
    return total / n


def tasa_en(paso, calentamiento=100, final=0.1):
    if paso < calentamiento:
        return TASA * (paso + 1) / calentamiento
    avance = (paso - calentamiento) / (PASOS - calentamiento)
    return TASA * (final + (1 - final) * 0.5 * (1 + math.cos(math.pi * avance)))


def entrenar(modelo, entrenamiento, validacion, intervalo=250):
    """AdamW con calendario; se queda con el mejor punto de validación."""
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=TASA,
                                    weight_decay=0.1)
    generador = torch.Generator().manual_seed(SEMILLA)
    historial, mejor, mejor_estado = [], math.inf, None
    for paso in range(PASOS + 1):
        if paso % intervalo == 0:
            medida = perdida_texto(modelo, validacion)
            en_entrenamiento = perdida_texto(modelo, entrenamiento[:200000])
            historial.append((paso, en_entrenamiento, medida))
            if medida < mejor:
                mejor = medida
                mejor_estado = {k: v.clone() for k, v
                                in modelo.state_dict().items()}
        if paso == PASOS:
            break
        for grupo in optimizador.param_groups:
            grupo["lr"] = tasa_en(paso)
        perdida = perdida_media(modelo, *lote(entrenamiento, generador))
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
    modelo.load_state_dict(mejor_estado)
    return historial

## 4. Generar

In [5]:
# --- 4. Generar (capítulo 8) -------------------------------------------------

@torch.no_grad()
def generar(modelo, simbolos, a_id, n, generador, largo=400, temperatura=0.8,
            p=0.9):
    """n relatos a la vez, con temperatura y top-p, empezando tras un salto."""
    contexto = modelo.config["contexto"]
    ids = torch.full((n, 1), a_id["\n"])
    for _ in range(largo):
        logits = modelo(ids[:, -contexto:])[:, -1]  # solo lo que cabe
        probabilidades = F.softmax(logits / temperatura, dim=-1)
        ordenadas, orden = probabilidades.sort(dim=-1, descending=True)
        antes = ordenadas.cumsum(dim=-1) - ordenadas
        ordenadas[antes >= p] = 0
        elegidos = torch.multinomial(ordenadas, 1, generator=generador)
        ids = torch.cat([ids, orden.gather(-1, elegidos)], dim=1)
    textos = ["".join(simbolos[i] for i in fila[1:]) for fila in ids.tolist()]
    return [texto.split("\n")[0] for texto in textos]

## 5. Evaluar

In [6]:
# --- 5. Evaluar (capítulo 10) ------------------------------------------------

PATRON = re.compile(
    r"(?:Había una vez (?P<art>una?) (?P<animal>\w+) (?P<adj>\w+) "
    r"(?P<llamado>llamad[oa]) (?P<nombre>\w+) que vivía en (?P<lugar>[^.]+)\."
    r"|En (?P<lugar2>[^.]+?) vivía (?P<art2>una?) (?P<animal2>\w+) "
    r"(?P<adj2>\w+) que se llamaba (?P<nombre2>\w+)\.)"
    r" (?:Un día|Una mañana), (?P<nombre3>\w+) (?:perdió|no encontraba) "
    r"su (?P<objeto>\w+)\."
    r" (?P<nombre4>\w+) (?P<pronombre>lo|la) buscó (?P<sitio1>[^,.]+), "
    r"pero no estaba allí\."
    r" Entonces (?P<ayudante>[^.]+?) le dijo que mirara (?P<sitio2>[^.]+)\."
    r" Allí estaba su (?P<objeto2>\w+), y (?P<nombre5>\w+) aprendió que "
    r"(?P<moraleja>[^.]+)\.")
CRITERIOS = ["estructura", "protagonista", "concordancia", "objeto",
             "sin inventos"]


def analizar(texto):
    """Comprueba un relato con criterios explícitos."""
    partes = PATRON.fullmatch(texto)
    if partes is None:
        return {"estructura": False}
    g = partes  # cada trozo con nombre; vale None si no ha encajado
    articulo = g["art"] or g["art2"]  # según la forma de empezar
    animal, nombre = g["animal"] or g["animal2"], g["nombre"] or g["nombre2"]
    adjetivo, lugar = g["adj"] or g["adj2"], g["lugar"] or g["lugar2"]
    llamado = "llamada" if articulo == "una" else "llamado"
    llamado_bien = g["llamado"] in (None, llamado)
    return {
        "estructura": True,
        "protagonista": nombre == g["nombre3"] == g["nombre4"] == g["nombre5"],
        "concordancia": ((articulo, animal) in ANIMALES and llamado_bien
                         and adjetivo in ADJETIVOS[articulo]
                         and nombre in NOMBRES[articulo]),
        "objeto": (g["objeto"] == g["objeto2"]
                   and OBJETOS.get(g["objeto"]) == g["pronombre"]),
        "sin inventos": (lugar in LUGARES and g["sitio1"] in SITIOS
                         and g["sitio2"] in SITIOS
                         and g["ayudante"] in AYUDANTES
                         and g["moraleja"] in MORALEJAS),
    }


def resumir(relatos, entrenamiento):
    """Porcentaje de relatos que cumplen cada criterio."""
    analisis = [analizar(texto) for texto in relatos]
    vistos = set(entrenamiento)
    resumen = {c: sum(a.get(c, False) for a in analisis) / len(relatos)
               for c in CRITERIOS}
    correctos = [t for t, a in zip(relatos, analisis)
                 if all(a.get(c, False) for c in CRITERIOS)]
    resumen["todo correcto"] = len(correctos) / len(relatos)
    resumen["nuevos entre los correctos"] = (
        sum(t not in vistos for t in correctos) / max(1, len(correctos)))
    return resumen


In [7]:
ejemplo = relato(random.Random(1))
print(ejemplo)
print(analizar(ejemplo))
perdido = PATRON.fullmatch(ejemplo)["objeto"]
otro = "farol" if perdido != "farol" else "mapa"
roto = ejemplo.replace("Allí estaba su " + perdido, "Allí estaba su " + otro)
print(analizar(roto))  # el objeto encontrado ya no es el perdido

En el bosque vivía una ardilla valiente que se llamaba Nina. Una mañana, Nina no encontraba su mapa. Nina lo buscó dentro de una bota vieja, pero no estaba allí. Entonces un caracol paciente le dijo que mirara entre las rocas. Allí estaba su mapa, y Nina aprendió que pedir ayuda no es de débiles.
{'estructura': True, 'protagonista': True, 'concordancia': True, 'objeto': True, 'sin inventos': True}
{'estructura': True, 'protagonista': True, 'concordancia': True, 'objeto': False, 'sin inventos': True}


## 6. Documentar

In [8]:
# --- 6. Documentar: ficha de datos y ficha del modelo ------------------------

def coma_decimal(texto):
    """Escribe los decimales con coma, como en el resto del libro."""
    return re.sub(r"(\d)\.(\d)", r"\1,\2", texto)


def protagonista_femenino(texto):
    primera_frase = texto.split(".")[0]
    return any(f"una {animal}" in primera_frase
               for articulo, animal in ANIMALES if articulo == "una")


def ficha_datos(entrenamiento, validacion, prueba, simbolos):
    todos = entrenamiento + validacion + prueba
    hembras = sum(protagonista_femenino(texto) for texto in todos)
    return coma_decimal("\n".join([
        "# Ficha de datos · Relatos de la gramática",
        "",
        "- **Motivación:** relatos sencillos cuya coherencia se puede "
        "comprobar con un programa.",
        "- **Origen:** generados por la función relato, con una gramática "
        "propia y la semilla %d. Sin datos personales ni textos de terceros."
        % SEMILLA,
        "- **Tamaño:** %d relatos distintos: %d de entrenamiento, %d de "
        "validación y %d de prueba, sin ningún relato repetido entre ellos."
        % (len(todos), len(entrenamiento), len(validacion), len(prueba)),
        "- **Longitud:** %.0f caracteres de media; %d símbolos distintos."
        % (sum(map(len, todos)) / len(todos), len(simbolos)),
        "- **Composición:** %d animales, %d nombres, %d objetos, "
        "%d moralejas; "
        "%.0f %% de protagonistas femeninas." % (
            len(ANIMALES), sum(map(len, NOMBRES.values())), len(OBJETOS),
            len(MORALEJAS), 100 * hembras / len(todos)),
        "- **Limitaciones:** todos los relatos siguen la misma estructura de "
        "cinco frases; el vocabulario es muy reducido.",
        "- **Uso previsto:** aprender a entrenar y evaluar un modelo pequeño.",
    ]))


def ficha_modelo(parametros, historial, prueba, resumen, segundos):
    peor = min(CRITERIOS, key=lambda criterio: resumen[criterio])
    return coma_decimal("\n".join([
        "# Ficha del modelo · Generador de relatos",
        "",
        "- **Arquitectura:** GPT de caracteres, %d bloques, %d cabezas, "
        "vectores de %d, contexto de %d caracteres; %s parámetros." % (
            CONFIG["capas"], CONFIG["cabezas"], CONFIG["dimension"],
            CONFIG["contexto"], f"{parametros:,}".replace(",", " ")),
        "- **Entrenamiento:** %d pasos con AdamW, tasa máxima %g con "
        "calentamiento y coseno, lotes de %d; %.0f segundos en CPU." % (
            PASOS, TASA, CONFIG["lote"], segundos),
        "- **Pérdidas:** validación %.3f en el mejor punto; prueba %.3f." % (
            min(h[2] for h in historial), prueba),
        "- **200 relatos generados** (temperatura 0,8 y top-p 0,9): "
        + "; ".join("%s %.1f %%" % (c, 100 * v) for c, v in resumen.items())
        + ".",
        "- **Fallo principal:** el criterio que menos relatos cumplen es "
        "«%s» (%.1f %%)." % (peor, 100 * resumen[peor]),
        "- **Uso previsto:** demostración docente. No sirve para escribir "
        "otros tipos de texto.",
        "- **Limitaciones:** solo conoce la gramática de sus datos; su "
        "contexto es de %d caracteres y los relatos miden unos 300."
        % CONFIG["contexto"],
    ]))

## 7. El proyecto completo

Entrena la segunda versión (contexto de 256 caracteres). Tarda unos dieciséis minutos. Para la primera versión, cambia `CONFIG` a `contexto=128` y `lote=32`.

In [9]:
# --- El proyecto completo ---------------------------------------------------

def ejecutar():
    torch.set_num_threads(4)
    torch.manual_seed(SEMILLA)
    entrenamiento, validacion, prueba = crear_datos()
    textos = {nombre: "\n".join(parte) + "\n" for nombre, parte in
              (("entrenamiento", entrenamiento), ("validación", validacion),
               ("prueba", prueba))}
    simbolos = sorted(set(textos["entrenamiento"]))
    a_id = {c: i for i, c in enumerate(simbolos)}
    datos = {nombre: torch.tensor([a_id[c] for c in texto])
             for nombre, texto in textos.items()}
    print("Relatos:", len(entrenamiento), len(validacion), len(prueba),
          "· símbolos:", len(simbolos))
    print("Ejemplo:", entrenamiento[0])

    modelo = GPT(len(simbolos), CONFIG)
    parametros = sum(p.numel() for p in modelo.parameters())
    inicio = time.perf_counter()
    historial = entrenar(modelo, datos["entrenamiento"], datos["validación"])
    segundos = time.perf_counter() - inicio
    perdida_prueba = perdida_texto(modelo, datos["prueba"])  # una sola vez
    print("Parámetros: %d · %.0f s · validación %.3f · prueba %.3f" % (
        parametros, segundos, min(h[2] for h in historial), perdida_prueba))
    print("Historial:", [(paso, round(e, 3), round(v, 3))
                         for paso, e, v in historial])

    generador = torch.Generator().manual_seed(SEMILLA)
    relatos = generar(modelo, simbolos, a_id, 200, generador)
    for texto in relatos[:3]:
        print("  ", texto)
    resumen = resumir(relatos, entrenamiento)
    print("Evaluación:", {c: round(v, 3) for c, v in resumen.items()})

    SALIDAS.mkdir(parents=True, exist_ok=True)
    (SALIDAS / "ficha_datos.md").write_text(
        ficha_datos(entrenamiento, validacion, prueba, simbolos),
        encoding="utf-8")
    (SALIDAS / "ficha_modelo.md").write_text(
        ficha_modelo(parametros, historial, perdida_prueba, resumen, segundos),
        encoding="utf-8")
    (SALIDAS / "relatos_generados.txt").write_text("\n".join(relatos),
                                                   encoding="utf-8")
    torch.save({"modelo": modelo.state_dict(), "config": CONFIG,
                "simbolos": simbolos}, SALIDAS / "relatos.pt")
    return {"historial": historial, "prueba": perdida_prueba,
            "resumen": resumen, "relatos": relatos[:10],
            "segundos": segundos, "parametros": parametros}

In [10]:
resultados = ejecutar()
print((SALIDAS / "ficha_modelo.md").read_text(encoding="utf-8"))

Relatos: 6400 800 800 · símbolos: 48
Ejemplo: En la montaña vivía un topo despistado que se llamaba Bruno. Una mañana, Bruno no encontraba su farol. Bruno lo buscó dentro de una bota vieja, pero no estaba allí. Entonces una abeja trabajadora le dijo que mirara en una cueva. Allí estaba su farol, y Bruno aprendió que los amigos aparecen cuando más se necesitan.
Parámetros: 221872 · 985 s · validación 0.098 · prueba 0.097
Historial: [(0, 4.097, 4.095), (250, 1.668, 1.667), (500, 0.242, 0.245), (750, 0.145, 0.146), (1000, 0.12, 0.12), (1250, 0.111, 0.111), (1500, 0.105, 0.106), (1750, 0.102, 0.102), (2000, 0.099, 0.1), (2250, 0.098, 0.098), (2500, 0.097, 0.098)]
   En un jardín vivía un zorro pequeño que se llamaba Lucas. Una mañana, Lucas no encontraba su libro. Lucas lo buscó bajo un puente, pero no estaba allí. Entonces una abeja trabajadora le dijo que mirara dentro de una bota vieja. Allí estaba su libro, y Lucas aprendió que pedir ayuda no es de débiles.
   En la montaña vivía un er

## Ejercicios

1. Comprueba que la gramática puede producir unos 97 millones de relatos distintos. ¿Qué fracción de ellos usamos para entrenar?

2. Añade a la gramática un animal nuevo, por ejemplo una cabra. ¿Qué tienes que cambiar en el programa para que la evaluación lo acepte?

3. Reproduce la primera versión cambiando `CONFIG` a un contexto de 128 y lotes de 32. Compara la pérdida, el tiempo y los criterios con los de la segunda versión.

4. Con `PATRON` y el método `start`, que da la posición donde empieza un trozo, calcula en los relatos de entrenamiento cuántos caracteres separan el nombre de la búsqueda del nombre final, y las dos menciones del objeto. ¿Caben en 128? ¿Y en 256?

5. Hay un fallo que nuestros criterios no detectan: que el ayudante recomiende buscar en el mismo sitio donde ya se buscó. Añade ese criterio a `analizar` y mide cuántos relatos de la segunda versión lo cumplen.

6. Con el modelo guardado en `salidas/contexto_256/relatos.pt`, genera 200 relatos con temperatura 0,5 y otros 200 con temperatura 1,2. ¿Cómo cambian los criterios? ¿Y la variedad de los relatos?

7. Propón una tercera versión que mejore el criterio del objeto sin cambiar la tarea. Por ejemplo, haz que muchos fragmentos de entrenamiento empiecen justo después de un salto de línea, al principio de un relato, para que la primera mención del objeto esté siempre a la vista cuando el modelo deba repetirlo; o usa los tokens BPE del capítulo 5 para que los relatos ocupen menos posiciones. Pruébala y compárala con las dos anteriores.

8. ¿Qué crees que pasaría si entrenáramos con solo 800 relatos? Piensa en la pérdida de validación, en los relatos correctos y en los nuevos, y explica qué te indicaría cada cifra.

9. Repasa la ficha de datos. Añade una sección de sesgos: ¿cómo se reparten los papeles en los relatos? ¿Hay algo que un lector de cuentos pudiera echar de menos?

10. Informe final. Escribe el informe final del proyecto respondiendo, en este orden, a las ocho preguntas de este capítulo. Incluye la ficha de datos, la ficha del modelo, una tabla de resultados de las dos versiones y un informe de fallos con criterios explícitos.

## Referencias

Inspirado en [nanochat](https://github.com/karpathy/nanochat), de Andrej Karpathy; en [TinyStories](https://arxiv.org/abs/2305.07759) (Eldan y Li, 2023); en [Datasheets for Datasets](https://arxiv.org/abs/1803.09010) (Gebru y colaboradores, 2021) y en [Model Cards for Model Reporting](https://arxiv.org/abs/1810.03993) (Mitchell y colaboradores, 2019). Código, gramática y experimentos propios.